### 1070. Product Sales Analysis III
```

Table: Sales

+-------------+-------+
| Column Name | Type  |
+-------------+-------+
| sale_id     | int   |
| product_id  | int   |
| year        | int   |
| quantity    | int   |
| price       | int   |
+-------------+-------+
(sale_id, year) is the primary key (combination of columns with unique values) of this table.
Each row records a sale of a product in a given year.
A product may have multiple sales entries in the same year.
Note that the per-unit price.

Write a solution to find all sales that occurred in the first year each product was sold.

For each product_id, identify the earliest year it appears in the Sales table.

Return all sales entries for that product in that year.

Return a table with the following columns: product_id, first_year, quantity, and price.
Return the result in any order.

 

Example 1:

Input: 
Sales table:
+---------+------------+------+----------+-------+
| sale_id | product_id | year | quantity | price |
+---------+------------+------+----------+-------+ 
| 1       | 100        | 2008 | 10       | 5000  |
| 2       | 100        | 2009 | 12       | 5000  |
| 7       | 200        | 2011 | 15       | 9000  |
+---------+------------+------+----------+-------+

Output: 
+------------+------------+----------+-------+
| product_id | first_year | quantity | price |
+------------+------------+----------+-------+ 
| 100        | 2008       | 10       | 5000  |
| 200        | 2011       | 15       | 9000  |
+------------+------------+----------+-------+


```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,IntegerType,StringType,StructField
from pyspark.sql.functions import col,min

In [2]:
spark=SparkSession.builder.getOrCreate()

26/08/02 20:04:33 WARN Utils: Your hostname, Chinmayas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.29.6 instead (on interface en0)
26/08/02 20:04:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/02 20:04:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'3.5.4'

In [4]:
data=[[1, 100, 2008, 10,5000], [2, 100, 2009, 12,5000], [7, 200, 2011, 15,9000]]
Schema=StructType([StructField("sale_id",IntegerType(),True)\
                   ,StructField("product_id",IntegerType(),True)\
                   ,StructField("year",IntegerType(),True)\
                   ,StructField("quantity",IntegerType(),True)\
                   ,StructField("price",IntegerType(),True)])
Sales=spark.createDataFrame(data,Schema)

In [5]:
Sales.show()

+-------+----------+----+--------+-----+
|sale_id|product_id|year|quantity|price|
+-------+----------+----+--------+-----+
|      1|       100|2008|      10| 5000|
|      2|       100|2009|      12| 5000|
|      7|       200|2011|      15| 9000|
+-------+----------+----+--------+-----+



In [15]:
Sales_min=Sales.groupBy("product_id").agg(min(col("year")).alias("first_year"))
Sales_min.show()

+----------+----------+
|product_id|first_year|
+----------+----------+
|       100|      2008|
|       200|      2011|
+----------+----------+



In [17]:
result=(

    Sales.alias("S").join(Sales_min.alias("SM"),col("S.year")==col("SM.first_year"),"inner")
        .select(col("S.product_id"),col("SM.first_year"),col("S.quantity"),col("S.price"))
)
result.show()

+----------+----------+--------+-----+
|product_id|first_year|quantity|price|
+----------+----------+--------+-----+
|       100|      2008|      10| 5000|
|       200|      2011|      15| 9000|
+----------+----------+--------+-----+



In [19]:
spark.stop()